# 03_DIM_SEASON

## Objective
This notebook creates and loads the Season Dimension for the Data Warehouse.

The season dimension classifies dates into meteorological seasons:
- Winter
- Spring
- Summer
- Autumn

## Data Source
The data is loaded from a manually created CSV file.

# Setup

In [ ]:
from pyspark.sql import SparkSession
from delta import configure_spark_with_delta_pip
from pyspark.sql.functions import col
import ConnectionConfig as cc
from delta.tables import DeltaTable, IdentityGenerator
from pyspark.sql.types import LongType, StringType

# Debug mode
debugging_mode = True

# Initialize environment


In [ ]:
cc.setupEnvironment()
spark = cc.startLocalCluster("DIM_SEASON")

spark.getActiveSession()

#  Load dim_season from CSV


In [ ]:
# PARAMETERS

csv_path = "data/Extra_Data/dim_season.csv"

In [ ]:
# EXTRACT
# Load season data from CSV

df_season = spark.read \
    .option("header", True) \
    .option("inferSchema", True) \
    .csv(csv_path)

In [ ]:
# DEBUG_CODE

if debugging_mode:
    df_season.show()

# TRANSFORM

In [ ]:
# TRANSFORM
dimSeason = df_season.createOrReplaceTempView("dimSeasonTemp")

df_dim_season = spark.sql("""
SELECT
  name AS season_name
FROM dimSeasonTemp
""")

In [ ]:
# DEBUG_CODE

if debugging_mode:
    df_dim_season.show()

# Create table with identity

In [ ]:
# Drop the old table if it exists
spark.sql("DROP TABLE IF EXISTS dim_season")

In [ ]:
cc.delete_delta_table(spark, "dim_season")

In [ ]:
# LOAD

# CREATE table with surrogate key
DeltaTable.create(spark) \
    .tableName("dim_season") \
    .addColumn("season_SK", LongType(), nullable=False, generatedAlwaysAs=IdentityGenerator(1,1)) \
    .addColumn("season_name", StringType(), nullable=False) \
    .property("delta.feature.identityColumns", "supported") \
    .execute()

# Writing the data to a delta-file


In [ ]:
# LOAD - Insert data

df_dim_season.write.format("delta").mode("append").saveAsTable("dim_season")


In [ ]:
# Delete parquet file if it already exists
cc.delete_parquet_table("parquet/dim_season_parquet")

In [ ]:
# Save as Parquet format

df = spark.read.format("delta").load("spark-warehouse/dim_season")

df.coalesce(1).write.format("parquet").save("./parquet/dim_season_parquet")

validation

In [ ]:
# DEBUG_CODE

if debugging_mode:
    spark.read.format("delta").load("spark-warehouse/dim_season").show()

cleanup

In [ ]:
# CLEANUP

spark.stop()

In [ ]:
%reset